# PulmoFoundation Model Loading Quickstart

This notebook demonstrates how to load the PulmoFoundation-E2 encoder and extract patch-level feature embeddings. It uses the importable `model_loading` package in this repository.

For meaningful embeddings, replace the demo image path with a real pathology patch, preferably a 512 x 512 RGB patch at 40X. If no image is provided, the notebook creates a synthetic RGB image only to verify that the loading and inference pipeline runs.

## 1. Prepare Paths

Run this notebook from the repository root or from the `notebooks/` directory. The PulmoFoundation checkpoint should be placed at `model_loading/ckpts/PulmoFoundation-E2.pth`.

In [ ]:
from pathlib import Path
import os
import sys

import torch
from PIL import Image, ImageDraw

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
    os.chdir(repo_root)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

checkpoint_path = repo_root / "model_loading" / "ckpts" / "PulmoFoundation-E2.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Repository root: {repo_root}")
print(f"Checkpoint path: {checkpoint_path}")
print(f"Device: {device}")

## 2. Load the Encoder and Transform

This step loads the Virchow2 backbone and the PulmoFoundation LoRA checkpoint. Access to the Virchow2 Hugging Face model may be required.

In [ ]:
from model_loading import get_model, get_transform

if not checkpoint_path.exists():
    raise FileNotFoundError(
        "PulmoFoundation checkpoint not found. Download PulmoFoundation-E2.pth "
        f"from Hugging Face and place it at: {checkpoint_path}"
    )

transform = get_transform()
model = get_model(device, str(checkpoint_path))

## 3. Load One Patch Image

Set `patch_path` to a real patch image if available. The fallback synthetic image is only for a plumbing check and should not be used for analysis.

In [ ]:
patch_path = Path("path/to/your/patch.jpg")

if patch_path.exists():
    image = Image.open(patch_path).convert("RGB")
    print(f"Loaded patch image: {patch_path}")
else:
    image = Image.new("RGB", (512, 512), color=(235, 230, 225))
    draw = ImageDraw.Draw(image)
    for idx in range(0, 512, 32):
        color = (180 + (idx % 50), 90 + (idx % 80), 125 + (idx % 60))
        draw.ellipse((idx, idx // 2, idx + 80, idx // 2 + 55), fill=color)
    print("Using a synthetic 512 x 512 RGB image for pipeline verification.")

image.size

## 4. Extract a Single Embedding

PulmoFoundation returns a 2560-dimensional embedding per input patch: 1280 dimensions from the class token concatenated with 1280 dimensions from mean-pooled patch tokens.

In [ ]:
image_tensor = transform(image).unsqueeze(0).to(device)
features = model(image_tensor)

print(f"Input tensor shape: {tuple(image_tensor.shape)}")
print(f"Feature tensor shape: {tuple(features.shape)}")
features[:1, :8]

## 5. Batch Extraction

For multiple patches, transform each image, stack them into a batch, and pass the batch through the loaded model.

In [ ]:
images = [image, image.copy()]
batch = torch.stack([transform(img) for img in images]).to(device)
batch_features = model(batch)

print(f"Batch tensor shape: {tuple(batch.shape)}")
print(f"Batch feature shape: {tuple(batch_features.shape)}")

## 6. Optional: Save Features

Downstream MIL workflows consume slide-level `.pt` tensors containing patch embeddings. The cell below is disabled by default to avoid writing files accidentally.

In [ ]:
SAVE_DEMO_FEATURES = False

if SAVE_DEMO_FEATURES:
    output_dir = repo_root / "tmp"
    output_dir.mkdir(exist_ok=True)
    output_path = output_dir / "demo_patch_features.pt"
    torch.save(batch_features.detach().cpu(), output_path)
    print(f"Saved demo features to: {output_path}")
else:
    print("Skipping feature save. Set SAVE_DEMO_FEATURES = True to write a demo .pt file.")

## Notes

- This notebook is for encoder loading and patch-level feature extraction.
- Whole-slide preprocessing, tissue detection, tiling, and feature extraction at scale should be handled with PrePATH.
- The released downstream diagnosis and survival workflows start from slide-level patch feature tensors under `TCGA__NSCLC/pt_files/PulmoFoundation-E2/`.